# 🧠 TRIBE v2 Experiment: Brain-State Similarity for EIT Scoring

**Experiment 1 (EXP-1)**  
**Target:** 038010_EIT-2A.mp3 (30 prompt-response pairs)

---

## 📋 Overview

This notebook validates the hypothesis: **"Same sounds → Same brain waves"**

### Hypothesis
If a learner produces a response similar to the native speaker prompt, TRIBE v2 should predict similar brain activation patterns for both. This experiment tests whether brain-state similarity can be used for meaning-based EIT scoring.

### Pipeline
```
Raw Audio → Apply start_offset → Segment → TRIBE v2 → Similarity → Visualization
```

### What This Notebook Does
1. **Environment Check** - GPU, CUDA, RAM validation
2. **Data Loading** - Audio + timestamps + stimuli
3. **EDA** - Waveform, spectrogram, audio analysis
4. **Audio Segmentation** - Split into prompt/response pairs
5. **TRIBE v2 Processing** - Generate brain predictions for all 60 segments
6. **Similarity Computation** - Cosine similarity, Pearson correlation
7. **Visualization** - Brain patterns, similarity heatmaps, high/low examples
8. **Analysis** - Qualitative examination of results

---

## 🚀 Instructions

1. **Upload to Colab:**
   - `038010_EIT-2A.mp3` (the audio file)
   - `038010_EIT-2A_timestamps.csv` (your timestamp annotations)
   
2. **Add Secrets in Colab:**
   - `HF_TOKEN` = your HuggingFace token (must have Llama 3.2 access)
   
3. **Enable GPU:**
   - Menu → Runtime → Change runtime type → GPU
   
4. **Run cells in order** - BUT note the important restart step below!

5. **IMPORTANT - Runtime Restart Required:**
   - After running the "Install TRIBE v2" cell, you MUST restart the runtime:
     - Go to **Runtime → Restart runtime** (or press Ctrl+M in Jupyter)
   - This is needed because TRIBE v2 upgrades numpy to 2.x which breaks librosa
   - After restart, re-run the verification cell and continue

---

**Estimated Runtime:** ~15-20 minutes for full processing

## 1. Environment Validation

Before processing, we validate the runtime environment to ensure GPU availability and compatible versions.

In [ ]:
# @title 🔧 Environment Check
import os
import sys
import subprocess
import warnings
warnings.filterwarnings('ignore')

print("=" * 60)
print("ENVIRONMENT VALIDATION")
print("=" * 60)

# Python version
print(f"\n🐍 Python Version: {sys.version}")

# CUDA availability
try:
    import torch
    print(f"🔥 PyTorch Version: {torch.__version__}")
    print(f"   CUDA Available: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"   CUDA Version: {torch.version.cuda}")
        print(f"   GPU: {torch.cuda.get_device_name(0)}")
        print(f"   GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    else:
        print("   ⚠️ WARNING: CUDA not available!")
except ImportError:
    print("   ❌ PyTorch not installed yet")

# nvidia-smi
try:
    result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
    if result.returncode == 0:
        print(f"\n📊 nvidia-smi:")
        lines = result.stdout.split('\n')
        for line in lines[:8]:
            print(f"   {line}")
except FileNotFoundError:
    print("   ⚠️ nvidia-smi not found")

# Memory check
try:
    import psutil
    mem = psutil.virtual_memory()
    print(f"\n💾 RAM: {mem.total / 1e9:.1f} GB total, {mem.available / 1e9:.1f} GB available")
except ImportError:
    print("   ⚠️ psutil not installed")

print("\n✅ Environment check complete!")

## 2. Installation

Install TRIBE v2 and required dependencies.

In [ ]:
# @title 📦 Install TRIBE v2 and Dependencies
import subprocess
import sys

print("Installing packages...")

# Install TRIBE v2 with plotting support
!uv pip install "tribev2[plotting] @ git+https://github.com/facebookresearch/tribev2.git"

# CRITICAL FIX: Downgrade numpy after TRIBE v2 installation
# TRIBE v2 upgrades numpy to 2.x which breaks librosa
# We need numpy==1.26.4 for librosa compatibility
print("\n🔧 Fixing numpy version for librosa compatibility...")
!uv pip install numpy==1.26.4 --force-reinstall

# Reinstall librosa to ensure it works with the downgraded numpy
!uv pip install librosa soundfile --force-reinstall

# Install additional dependencies
!uv pip install pandas matplotlib seaborn scikit-learn tqdm psutil

print("\n✅ Installation complete!")
print("   Note: If you see import errors below, go to Runtime → Restart runtime")
print("   then re-run this cell to verify the fix worked.")

In [ ]:
# @title 🔄 Restart Runtime (Required after numpy fix)
# This cell restarts the Python kernel to pick up the new numpy version
import sys
import subprocess

print("=" * 60)
print("RESTARTING RUNTIME")
print("=" * 60)
print("\n⚠️  IMPORTANT: Please manually restart the runtime!")
print("   Go to: Runtime → Restart runtime")
print("   Then re-run this cell and all subsequent cells.")
print("\n   Or click the button below if available:")

# Check numpy version before restart
try:
    import numpy as np
    print(f"\n   Current numpy version: {np.__version__}")
    if np.__version__.startswith('2.'):
        print("   ❌ numpy is still version 2.x - restart required!")
    else:
        print("   ✅ numpy version is compatible")
except ImportError as e:
    print(f"   ⚠️ numpy import error: {e}")
    print("   Restart required!")

print("\n⏸️  After restart, continue running cells from where you left off.")

In [ ]:
# @title ✅ Verify Environment After Restart
import sys
import subprocess

print("=" * 60)
print("VERIFYING ENVIRONMENT")
print("=" * 60)

# Check numpy version
try:
    import numpy as np
    print(f"\n✅ numpy: {np.__version__}")
    if np.__version__.startswith('2.'):
        print("   ❌ ERROR: numpy is version 2.x - this will break librosa!")
        print("   Run the installation cell again and restart.")
except ImportError as e:
    print(f"❌ numpy import failed: {e}")

# Check librosa
try:
    import librosa
    print(f"✅ librosa: {librosa.__version__}")
except ImportError as e:
    print(f"❌ librosa import failed: {e}")

# Check soundfile
try:
    import soundfile as sf
    print(f"✅ soundfile: installed")
except ImportError as e:
    print(f"❌ soundfile import failed: {e}")

# Check TRIBE v2
try:
    from tribev2.demo_utils import TribeModel
    print(f"✅ tribev2: installed")
except ImportError as e:
    print(f"❌ tribev2 import failed: {e}")

print("\n✅ Environment verification complete! Continue to next cell.")

## 3. Authentication & Configuration

Authenticate with HuggingFace and set up paths.

In [ ]:
# @title 🔐 Authenticate with HuggingFace
from huggingface_hub import login
from google.colab import userdata

# Get token from Colab Secrets
hf_token = userdata.get("HF_TOKEN")

if hf_token:
    login(token=hf_token)
    print("✅ HuggingFace authenticated!")
else:
    print("❌ HF_TOKEN not found in Colab Secrets!")
    print("   Please add HF_TOKEN to your Colab secrets.")

In [ ]:
# @title 📁 Setup Paths
import os
from pathlib import Path

print("=" * 60)
print("PATH CONFIGURATION")
print("=" * 60)

# Working directory
WORK_DIR = Path("/content/EXP-1")
WORK_DIR.mkdir(parents=True, exist_ok=True)

# Audio and data directories
AUDIO_DIR = Path("/content")
CACHE_DIR = Path("/content/cache")
CACHE_DIR.mkdir(parents=True, exist_ok=True)

# Find uploaded files
audio_files = list(AUDIO_DIR.glob('*.mp3')) + list(AUDIO_DIR.glob('*.wav'))
csv_files = list(AUDIO_DIR.glob('*.csv'))

print(f"\n📂 Working Directory: {WORK_DIR}")
print(f"\n📄 Found {len(audio_files)} audio file(s):")
for f in audio_files:
    print(f"   • {f.name}")

print(f"\n📊 Found {len(csv_files)} CSV file(s):")
for f in csv_files:
    print(f"   • {f.name}")

# Create subdirectories
for subdir in ['audio/prompt', 'audio/response', 'predictions/prompt', 'predictions/response', 'visualizations']:
    (WORK_DIR / subdir).mkdir(parents=True, exist_ok=True)

print("\n✅ Directory structure created!")

## 4. Data Loading

Load the audio file, timestamps CSV, and validate the data.

In [ ]:
# @title 🎵 Load Audio File
import librosa
import soundfile as sf
import numpy as np
import pandas as pd

print("=" * 60)
print("LOADING AUDIO")
print("=" * 60)

# Configuration
AUDIO_FILE = "038010_EIT-2A.mp3"
START_OFFSET_SEC = 150  # Skip English instructions

audio_path = AUDIO_DIR / AUDIO_FILE

if not audio_path.exists():
    print(f"❌ File not found: {AUDIO_FILE}")
    print("   Please upload the audio file to Colab!")
else:
    print(f"\n📄 Loading: {audio_path.name}")
    y, sr = librosa.load(audio_path, sr=None)
    duration = len(y) / sr
    
    print(f"   Duration: {duration:.1f}s ({duration/60:.2f} min)")
    print(f"   Sample Rate: {sr} Hz")
    print(f"   Samples: {len(y):,}")
    print(f"   Channels: {1 if len(y.shape) == 1 else y.shape[1]}")
    print(f"   Start Offset: {START_OFFSET_SEC}s (English instructions)")
    print(f"   Useful Audio: {duration - START_OFFSET_SEC:.1f}s")
    
    # Save as WAV for TRIBE v2
    wav_path = WORK_DIR / "audio_full.wav"
    sf.write(wav_path, y, sr)
    print(f"\n💾 Saved as WAV: {wav_path.name}")

In [ ]:
# @title 📋 Load Timestamps & Stimuli
print("=" * 60)
print("LOADING TIMESTAMPS")
print("=" * 60)

# Find timestamps CSV
timestamp_file = None
for f in csv_files:
    if 'timestamp' in f.name.lower():
        timestamp_file = f
        break

if timestamp_file is None:
    print("❌ Timestamp CSV not found!")
    print("   Please upload 038010_EIT-2A_timestamps.csv")
else:
    print(f"\n📄 Loading: {timestamp_file.name}")
    timestamps_df = pd.read_csv(timestamp_file)
    
    print(f"   Columns: {list(timestamps_df.columns)}")
    print(f"   Rows: {len(timestamps_df)}")
    
    # Validate required columns
    required_cols = ['item_number', 'stimulus', 'prompt_start_sec', 'prompt_end_sec', 'learner_start_sec', 'learner_end_sec']
    missing = [c for c in required_cols if c not in timestamps_df.columns]
    
    if missing:
        print(f"\n⚠️ Missing columns: {missing}")
    else:
        print(f"\n✅ All required columns present")
    
    # Check for empty timestamps
    empty_prompt = timestamps_df['prompt_start_sec'].isna().sum()
    empty_learner = timestamps_df['learner_start_sec'].isna().sum()
    print(f"\n   Empty prompt timestamps: {empty_prompt}")
    print(f"   Empty learner timestamps: {empty_learner}")
    
    # Display sample
    print(f"\n📊 Sample data:")
    display(timestamps_df.head(5))
    
    # Validate timestamps: check for time order issues
    print(f"
🔍 Validating timestamp order...")
    time_errors = []
    for idx, row in timestamps_df.iterrows():
        item = row['item_number']
        p_start = row['prompt_start_sec']
        p_end = row['prompt_end_sec']
        l_start = row['learner_start_sec']
        l_end = row['learner_end_sec']
        
        if pd.isna(p_start) or pd.isna(p_end) or pd.isna(l_start) or pd.isna(l_end):
            continue
            
        if p_start >= p_end:
            time_errors.append(f"Item {item}: prompt_start ({p_start}) >= prompt_end ({p_end})")
        if l_start >= l_end:
            time_errors.append(f"Item {item}: learner_start ({l_start}) >= learner_end ({l_end})")
        if p_end > l_start:
            time_errors.append(f"Item {item}: prompt_end ({p_end}) > learner_start ({l_start}) - possible overlap")
    
    if time_errors:
        print(f"
⚠️ Timestamp order issues found:")
        for err in time_errors:
            print(f"   • {err}")
    else:
        print(f"
✅ All timestamps are in valid order")

In [ ]:
# @title 📝 Load Stimuli (Reference Sentences)
# Using stimuli from the CSV file (already contains correct sentences)

print(f"✅ Using {len(timestamps_df)} stimulus sentences from CSV")

display(timestamps_df[['item_number', 'stimulus']].head(3))

## 5. Exploratory Data Analysis (EDA)

Visualize the full audio to understand its structure before segmentation.

In [ ]:
# @title 📈 Full Audio Waveform
import matplotlib.pyplot as plt
import seaborn as sns

print("=" * 60)
print("WAVEFORM VISUALIZATION")
print("=" * 60)

fig, ax = plt.subplots(figsize=(16, 4))

# Downsample for visualization
downsample = max(1, len(y) // 5000)
y_ds = y[::downsample]
time = np.arange(len(y_ds)) * downsample / sr

ax.fill_between(time, y_ds, alpha=0.5, color='steelblue')
ax.axvline(x=START_OFFSET_SEC, color='red', linestyle='--', linewidth=2, label=f'Start offset ({START_OFFSET_SEC}s)')
ax.set_xlim(0, time[-1])
ax.set_xlabel('Time (seconds)', fontsize=12)
ax.set_ylabel('Amplitude', fontsize=12)
ax.set_title(f'Full Audio Waveform: {audio_path.name}\n(Duration: {duration:.1f}s = {duration/60:.1f} min)', fontsize=14, fontweight='bold')
ax.legend(loc='upper right')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(WORK_DIR / 'visualizations/waveform_full.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"✅ Saved: waveform_full.png")

In [ ]:
# @title 🌊 Full Audio Spectrogram
print("=" * 60)
print("SPECTROGRAM VISUALIZATION")
print("=" * 60)

fig, ax = plt.subplots(figsize=(16, 5))

D = librosa.stft(y)
D_db = librosa.amplitude_to_db(np.abs(D), ref=np.max)

img = librosa.display.specshow(D_db, sr=sr, x_axis='time', y_axis='hz', ax=ax, cmap='viridis')
ax.axvline(x=START_OFFSET_SEC, color='red', linestyle='--', linewidth=2, label=f'Start offset ({START_OFFSET_SEC}s)')
ax.set_title(f'Full Audio Spectrogram: {audio_path.name}', fontsize=14, fontweight='bold')
ax.set_xlabel('Time (seconds)', fontsize=12)
ax.set_ylabel('Frequency (Hz)', fontsize=12)
fig.colorbar(img, ax=ax, format='%+2.0f dB', label='dB')
ax.legend(loc='upper right')

plt.tight_layout()
plt.savefig(WORK_DIR / 'visualizations/spectrogram_full.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"✅ Saved: spectrogram_full.png")

In [ ]:
# @title 🔊 Audio Analysis: Energy Distribution
print("=" * 60)
print("ENERGY ANALYSIS")
print("=" * 60)

# Compute RMS energy
frame_length = int(0.025 * sr)  # 25ms frames
hop_length = int(0.010 * sr)    # 10ms hop
rms = librosa.feature.rms(y=y, frame_length=frame_length, hop_length=hop_length)[0]
times = librosa.times_like(rms, sr=sr, hop_length=hop_length)

fig, axes = plt.subplots(2, 1, figsize=(16, 6))

# Plot 1: Energy over time
ax1 = axes[0]
ax1.fill_between(times, rms, alpha=0.5, color='coral')
ax1.axvline(x=START_OFFSET_SEC, color='red', linestyle='--', linewidth=2, label='Start Offset')
ax1.set_xlabel('Time (seconds)')
ax1.set_ylabel('RMS Energy')
ax1.set_title('Audio Energy Over Time', fontsize=12, fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Plot 2: Energy histogram
ax2 = axes[1]
ax2.hist(rms, bins=50, color='coral', alpha=0.7, edgecolor='black')
ax2.axvline(x=np.mean(rms), color='blue', linestyle='-', linewidth=2, label=f'Mean: {np.mean(rms):.3f}')
ax2.axvline(x=np.median(rms), color='green', linestyle='-', linewidth=2, label=f'Median: {np.median(rms):.3f}')
ax2.set_xlabel('RMS Energy')
ax2.set_ylabel('Frequency')
ax2.set_title('Energy Distribution', fontsize=12, fontweight='bold')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(WORK_DIR / 'visualizations/energy_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

# Statistics
print(f"\n📊 Energy Statistics (after {START_OFFSET_SEC}s):")
useful_start = int(START_OFFSET_SEC * sr)
rms_useful = rms[times > START_OFFSET_SEC]
print(f"   Mean: {np.mean(rms_useful):.4f}")
print(f"   Std:  {np.std(rms_useful):.4f}")
print(f"   Min:  {np.min(rms_useful):.4f}")
print(f"   Max:  {np.max(rms_useful):.4f}")
print(f"\n✅ Saved: energy_analysis.png")

## 6. Audio Segmentation

Segment the audio into prompt and response pairs based on timestamps.

In [ ]:
# @title ✂️ Segment Audio into Prompt/Response Pairs
from tqdm import tqdm

print("=" * 60)
print("AUDIO SEGMENTATION")
print("=" * 60)

# Load full audio
y_full, sr = librosa.load(audio_path, sr=None)

segmented_items = []
errors = []

print(f"\nProcessing {len(timestamps_df)} items...")

for idx, row in tqdm(timestamps_df.iterrows(), total=len(timestamps_df)):
    item_num = row['item_number']
    
    # Get timestamps (ensure they're floats)
    try:
        p_start = float(row['prompt_start_sec'])
        p_end = float(row['prompt_end_sec'])
        l_start = float(row['learner_start_sec'])
        l_end = float(row['learner_end_sec'])
        
        # Validate
        if any(pd.isna([p_start, p_end, l_start, l_end])):
            errors.append(f"Item {item_num}: Missing timestamps")
            continue
        
        if p_start >= p_end or l_start >= l_end:
            errors.append(f"Item {item_num}: Invalid time order")
            continue
        
        # Extract prompt audio
        p_start_idx = int(p_start * sr)
        p_end_idx = int(p_end * sr)
        y_prompt = y_full[p_start_idx:p_end_idx]
        
        # Extract learner audio
        l_start_idx = int(l_start * sr)
        l_end_idx = int(l_end * sr)
        y_learner = y_full[l_start_idx:l_end_idx]
        
        # Save as WAV
        prompt_path = WORK_DIR / 'audio/prompt' / f'prompt_{item_num:02d}.wav'
        response_path = WORK_DIR / 'audio/response' / f'response_{item_num:02d}.wav'
        
        sf.write(prompt_path, y_prompt, sr)
        sf.write(response_path, y_learner, sr)
        
        segmented_items.append({
            'item_number': item_num,
            'stimulus': row['stimulus'],
            'prompt_path': str(prompt_path),
            'response_path': str(response_path),
            'prompt_duration': len(y_prompt) / sr,
            'response_duration': len(y_learner) / sr,
            'prompt_start': p_start,
            'prompt_end': p_end,
            'learner_start': l_start,
            'learner_end': l_end
        })
        
    except Exception as e:
        errors.append(f"Item {item_num}: {str(e)}")

# Create DataFrame
segmented_df = pd.DataFrame(segmented_items)

print(f"\n✅ Successfully segmented: {len(segmented_df)} items")
if errors:
    print(f"\n⚠️ Errors ({len(errors)}):")
    for err in errors[:5]:
        print(f"   • {err}")

print(f"\n📊 Segment Duration Statistics:")
print(f"   Prompt - Mean: {segmented_df['prompt_duration'].mean():.2f}s, Std: {segmented_df['prompt_duration'].std():.2f}s")
print(f"   Response - Mean: {segmented_df['response_duration'].mean():.2f}s, Std: {segmented_df['response_duration'].std():.2f}s")
print(f"\n📁 Saved to: {WORK_DIR}/audio/{prompt,response}/")

In [ ]:
# @title 🎵 Sample Segmented Audio
print("=" * 60)
print("SAMPLE SEGMENTED AUDIO")
print("=" * 60)

# Show first 5 items
fig, axes = plt.subplots(5, 2, figsize=(14, 12))

for i, row in segmented_df.head(5).iterrows():
    item = int(row['item_number'])
    
    # Load prompt
    y_p, sr_p = librosa.load(row['prompt_path'], sr=None)
    t_p = np.arange(len(y_p)) / sr_p
    
    # Load response
    y_r, sr_r = librosa.load(row['response_path'], sr=None)
    t_r = np.arange(len(y_r)) / sr_r
    
    # Plot prompt
    axes[i, 0].fill_between(t_p, y_p, alpha=0.5, color='steelblue')
    axes[i, 0].set_title(f'Item {item}: Prompt ({row["prompt_duration"]:.2f}s)', fontsize=10)
    axes[i, 0].set_ylabel('Amplitude')
    
    # Plot response
    axes[i, 1].fill_between(t_r, y_r, alpha=0.5, color='coral')
    axes[i, 1].set_title(f'Item {item}: Response ({row["response_duration"]:.2f}s)', fontsize=10)
    axes[i, 1].set_ylabel('Amplitude')

for ax in axes[-1]:
    ax.set_xlabel('Time (s)')

plt.suptitle('Sample Segmented Audio: Prompt (blue) vs Response (coral)', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(WORK_DIR / 'visualizations/sample_segments.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"\n✅ Saved: sample_segments.png")

## 7. TRIBE v2 Processing

Load the TRIBE v2 model and generate brain predictions for all segments.

In [ ]:
# @title 🧠 Load TRIBE v2 Model
from tribev2.demo_utils import TribeModel

print("=" * 60)
print("LOADING TRIBE v2 MODEL")
print("=" * 60)

print("\nLoading TRIBE v2 model from HuggingFace...")
print("   (This may take a minute on first run)")

model = TribeModel.from_pretrained(
    "facebook/tribev2",
    cache_folder=CACHE_DIR,
)

print("\n✅ TRIBE v2 Model loaded!")
print(f"   Model type: {type(model).__name__}")
print(f"   Cache: {CACHE_DIR}")

In [ ]:
# @title 🧠 Generate Brain Predictions (Progress Tracked)
from tqdm import tqdm
import time

print("=" * 60)
print("TRIBE v2 PROCESSING")
print("=" * 60)

def process_audio_with_tribe(audio_path, model):
    """Process audio file through TRIBE v2 and return predictions."""
    try:
        df = model.get_events_dataframe(audio_path=str(audio_path))
        preds, segments = model.predict(events=df, verbose=False)
        return preds, segments
    except Exception as e:
        print(f"   Error processing {audio_path}: {e}")
        return None, None

results = []
total_items = len(segmented_df) * 2  # prompt + response

print(f"\nProcessing {len(segmented_df)} prompt + {len(segmented_df)} response audio files...")
print(f"Total: {total_items} files")

start_time = time.time()

# Process all segments
for idx, row in tqdm(segmented_df.iterrows(), total=len(segmented_df), desc="Items"):
    item = int(row['item_number'])
    
    # Process prompt
    prompt_path = row['prompt_path']
    prompt_preds, prompt_segments = process_audio_with_tribe(prompt_path, model)
    
    if prompt_preds is not None:
        # Save predictions
        np.save(WORK_DIR / 'predictions/prompt' / f'item_{item:02d}.npy', prompt_preds)
    else:
        prompt_preds = np.array([])
    
    # Process response
    response_path = row['response_path']
    response_preds, response_segments = process_audio_with_tribe(response_path, model)
    
    if response_preds is not None:
        # Save predictions
        np.save(WORK_DIR / 'predictions/response' / f'item_{item:02d}.npy', response_preds)
    else:
        response_preds = np.array([])
    
    # Store result metadata
    results.append({
        'item': item,
        'stimulus': row['stimulus'],
        'prompt_shape': prompt_preds.shape if prompt_preds.size > 0 else None,
        'response_shape': response_preds.shape if response_preds.size > 0 else None,
    })

elapsed = time.time() - start_time
results_df = pd.DataFrame(results)

print(f"\n⏱️ Total processing time: {elapsed/60:.1f} minutes")
print(f"   Average per item: {elapsed/len(segmented_df)/60:.2f} minutes")
print(f"\n✅ Predictions saved to: {WORK_DIR}/predictions/")

display(results_df.head(10))

## 8. Similarity Computation

Compute brain-state similarity between prompt and response for each item.

In [ ]:
# @title 📊 Compute Brain-State Similarity
from sklearn.metrics.pairwise import cosine_similarity
from scipy.stats import pearsonr, spearmanr
import numpy as np

print("=" * 60)
print("SIMILARITY COMPUTATION")
print("=" * 60)

def compute_similarities(prompt_preds, response_preds):
    """
    Compute multiple similarity metrics between two brain prediction arrays.
    
    Returns:
    - cosine: Cosine similarity (whole-brain)
    - pearson: Pearson correlation coefficient
    """
    if prompt_preds is None or response_preds is None:
        return {'cosine': np.nan, 'pearson': np.nan}
    
    if prompt_preds.size == 0 or response_preds.size == 0:
        return {'cosine': np.nan, 'pearson': np.nan}
    
    # Average over time for each
    prompt_avg = np.mean(prompt_preds, axis=0)
    response_avg = np.mean(response_preds, axis=0)
    
    # Cosine similarity
    cos_sim = cosine_similarity([prompt_avg], [response_avg])[0, 0]
    
    # Pearson correlation
    try:
        pearson_corr, _ = pearsonr(prompt_avg, response_avg)
    except:
        pearson_corr = np.nan
    
    return {
        'cosine': cos_sim,
        'pearson': pearson_corr
    }

similarity_results = []

print(f"\nComputing similarities for {len(segmented_df)} items...")

for idx, row in tqdm(segmented_df.iterrows(), total=len(segmented_df), desc="Computing"):
    item = int(row['item_number'])
    
    # Load predictions
    prompt_preds = np.load(WORK_DIR / 'predictions/prompt' / f'item_{item:02d}.npy')
    response_preds = np.load(WORK_DIR / 'predictions/response' / f'item_{item:02d}.npy')
    
    # Compute similarities
    sims = compute_similarities(prompt_preds, response_preds)
    
    similarity_results.append({
        'item_number': item,
        'stimulus': row['stimulus'],
        'prompt_duration': row['prompt_duration'],
        'response_duration': row['response_duration'],
        'cosine_similarity': sims['cosine'],
        'pearson_correlation': sims['pearson']
    })

similarity_df = pd.DataFrame(similarity_results)

print(f"\n✅ Similarity computation complete!")
print(f"\n📊 Similarity Statistics:")
print(f"   Cosine Similarity:")
print(f"      Mean: {similarity_df['cosine_similarity'].mean():.4f}")
print(f"      Std:  {similarity_df['cosine_similarity'].std():.4f}")
print(f"      Min:  {similarity_df['cosine_similarity'].min():.4f}")
print(f"      Max:  {similarity_df['cosine_similarity'].max():.4f}")
print(f"\n   Pearson Correlation:")
print(f"      Mean: {similarity_df['pearson_correlation'].mean():.4f}")
print(f"      Std:  {similarity_df['pearson_correlation'].std():.4f}")

# Save results
similarity_df.to_csv(WORK_DIR / 'similarity_results.csv', index=False)
print(f"\n💾 Saved: similarity_results.csv")

## 9. Visualizations

Comprehensive visualizations to understand the brain-state similarity patterns.

In [ ]:
# @title 📈 Similarity Distribution
print("=" * 60)
print("SIMILARITY DISTRIBUTION")
print("=" * 60)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Cosine similarity histogram
ax1 = axes[0]
ax1.hist(similarity_df['cosine_similarity'].dropna(), bins=15, color='steelblue', alpha=0.7, edgecolor='black')
ax1.axvline(x=similarity_df['cosine_similarity'].mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {similarity_df["cosine_similarity"].mean():.3f}')
ax1.set_xlabel('Cosine Similarity', fontsize=12)
ax1.set_ylabel('Frequency', fontsize=12)
ax1.set_title('Cosine Similarity Distribution', fontsize=12, fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Pearson correlation histogram
ax2 = axes[1]
ax2.hist(similarity_df['pearson_correlation'].dropna(), bins=15, color='coral', alpha=0.7, edgecolor='black')
ax2.axvline(x=similarity_df['pearson_correlation'].mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {similarity_df["pearson_correlation"].mean():.3f}')
ax2.set_xlabel('Pearson Correlation', fontsize=12)
ax2.set_ylabel('Frequency', fontsize=12)
ax2.set_title('Pearson Correlation Distribution', fontsize=12, fontweight='bold')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(WORK_DIR / 'visualizations/similarity_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"✅ Saved: similarity_distribution.png")

In [ ]:
# @title 🔥 Similarity Heatmap
print("=" * 60)
print("SIMILARITY HEATMAP")
print("=" * 60)

# Create similarity matrix
n_items = len(similarity_df)
sim_matrix = np.zeros((n_items, n_items))

for i, row_i in similarity_df.iterrows():
    for j, row_j in similarity_df.iterrows():
        sim_matrix[i, j] = row_i['cosine_similarity']

fig, ax = plt.subplots(figsize=(12, 10))

im = ax.imshow(sim_matrix, cmap='viridis', aspect='auto')
ax.set_xlabel('Item Number', fontsize=12)
ax.set_ylabel('Item Number', fontsize=12)
ax.set_title('Brain-State Similarity Matrix\n(Cosine Similarity between Prompt-Response Pairs)', fontsize=14, fontweight='bold')

# Add colorbar
cbar = plt.colorbar(im, ax=ax)
cbar.set_label('Cosine Similarity', fontsize=12)

plt.tight_layout()
plt.savefig(WORK_DIR / 'visualizations/similarity_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"✅ Saved: similarity_heatmap.png")

In [ ]:
# @title 🏆 Top 5 Highest Similarity Items
print("=" * 60)
print("TOP 5 HIGHEST SIMILARITY ITEMS")
print("=" * 60)

top5_high = similarity_df.nlargest(5, 'cosine_similarity')

print("\nThese items show the HIGHEST brain-state similarity between prompt and response.")
print("Interpretation: TRIBE v2 sees these responses as most similar to the native prompt.")
print()

display(top5_high[['item_number', 'stimulus', 'cosine_similarity', 'pearson_correlation']])

print("\n📝 Details:")
for _, row in top5_high.iterrows():
    print(f"\n   Item {row['item_number']}:")
    print(f"   Stimulus: {row['stimulus']}")
    print(f"   Cosine: {row['cosine_similarity']:.4f}, Pearson: {row['pearson_correlation']:.4f}")

In [ ]:
# @title ⚠️ Top 5 Lowest Similarity Items
print("=" * 60)
print("TOP 5 LOWEST SIMILARITY ITEMS")
print("=" * 60)

top5_low = similarity_df.nsmallest(5, 'cosine_similarity')

print("\nThese items show the LOWEST brain-state similarity between prompt and response.")
print("Interpretation: TRIBE v2 sees these responses as most different from the native prompt.")
print()

display(top5_low[['item_number', 'stimulus', 'cosine_similarity', 'pearson_correlation']])

print("\n📝 Details:")
for _, row in top5_low.iterrows():
    print(f"\n   Item {row['item_number']}:")
    print(f"   Stimulus: {row['stimulus']}")
    print(f"   Cosine: {row['cosine_similarity']:.4f}, Pearson: {row['pearson_correlation']:.4f}")

## 10. Brain Activation Visualization

Visualize brain activation patterns for prompt vs response side-by-side.

In [ ]:
# @title 🧠 Brain Activation: High Similarity Example (3D Surface)
from tribev2.plotting import PlotBrain

print("=" * 60)
print("BRAIN ACTIVATION - HIGH SIMILARITY EXAMPLE (3D Surface)")
print("=" * 60)

# Get highest similarity item
top_item = int(top5_high.iloc[0]['item_number'])
print(f"\nShowing Item {top_item}: {top5_high.iloc[0]['stimulus']}")
print(f"Cosine Similarity: {top5_high.iloc[0]['cosine_similarity']:.4f}")

# Load predictions
prompt_preds = np.load(WORK_DIR / 'predictions/prompt' / f'item_{top_item:02d}.npy')
response_preds = np.load(WORK_DIR / 'predictions/response' / f'item_{top_item:02d}.npy')

print(f"\n   Prompt predictions shape: {prompt_preds.shape}")
print(f"   Response predictions shape: {response_preds.shape}")

# Initialize plotter
plotter = PlotBrain(mesh="fsaverage5")

# Get number of timesteps to show
n_timesteps = min(5, prompt_preds.shape[0], response_preds.shape[0])
print(f"\n   Rendering {n_timesteps} timesteps...")

# Plot prompt brain activation
print("\n   Rendering Prompt brain...")
fig_prompt = plotter.plot_timesteps(
    prompt_preds[:n_timesteps],
    cmap="fire",
    norm_percentile=99,
    vmin=.6,
    alpha_cmap=(0, .2),
    show_stimuli=False
)
plt.suptitle(f'Item {top_item} - Prompt (High Similarity)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(WORK_DIR / 'visualizations/brain_3d_high_prompt.png', dpi=150, bbox_inches='tight')
plt.show()

# Plot response brain activation
print("\n   Rendering Response brain...")
fig_response = plotter.plot_timesteps(
    response_preds[:n_timesteps],
    cmap="fire",
    norm_percentile=99,
    vmin=.6,
    alpha_cmap=(0, .2),
    show_stimuli=False
)
plt.suptitle(f'Item {top_item} - Response (High Similarity)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(WORK_DIR / 'visualizations/brain_3d_high_response.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"\n✅ Saved: brain_3d_high_prompt.png, brain_3d_high_response.png")

In [ ]:
# @title 🧠 Brain Activation: Low Similarity Example (3D Surface)
print("=" * 60)
print("BRAIN ACTIVATION - LOW SIMILARITY EXAMPLE (3D Surface)")
print("=" * 60)

# Get lowest similarity item
low_item = int(top5_low.iloc[0]['item_number'])
print(f"\nShowing Item {low_item}: {top5_low.iloc[0]['stimulus']}")
print(f"Cosine Similarity: {top5_low.iloc[0]['cosine_similarity']:.4f}")

# Load predictions
prompt_preds_low = np.load(WORK_DIR / 'predictions/prompt' / f'item_{low_item:02d}.npy')
response_preds_low = np.load(WORK_DIR / 'predictions/response' / f'item_{low_item:02d}.npy')

print(f"\n   Prompt predictions shape: {prompt_preds_low.shape}")
print(f"   Response predictions shape: {response_preds_low.shape}")

# Initialize plotter
plotter = PlotBrain(mesh="fsaverage5")

# Get number of timesteps to show
n_timesteps = min(5, prompt_preds_low.shape[0], response_preds_low.shape[0])
print(f"\n   Rendering {n_timesteps} timesteps...")

# Plot prompt brain activation
print("\n   Rendering Prompt brain...")
fig_prompt = plotter.plot_timesteps(
    prompt_preds_low[:n_timesteps],
    cmap="fire",
    norm_percentile=99,
    vmin=.6,
    alpha_cmap=(0, .2),
    show_stimuli=False
)
plt.suptitle(f'Item {low_item} - Prompt (Low Similarity)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(WORK_DIR / 'visualizations/brain_3d_low_prompt.png', dpi=150, bbox_inches='tight')
plt.show()

# Plot response brain activation
print("\n   Rendering Response brain...")
fig_response = plotter.plot_timesteps(
    response_preds_low[:n_timesteps],
    cmap="fire",
    norm_percentile=99,
    vmin=.6,
    alpha_cmap=(0, .2),
    show_stimuli=False
)
plt.suptitle(f'Item {low_item} - Response (Low Similarity)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(WORK_DIR / 'visualizations/brain_3d_low_response.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"\n✅ Saved: brain_3d_low_prompt.png, brain_3d_low_response.png")

In [ ]:
# @title 📊 Brain Activation Time Series Comparison
print("=" * 60)
print("BRAIN ACTIVATION TIME SERIES COMPARISON")
print("=" * 60)

fig, axes = plt.subplots(3, 2, figsize=(14, 10))

examples = [
    (top_item, "High Similarity"),
    (low_item, "Low Similarity"),
    (int(segmented_df.iloc[14]['item_number']), "Medium Similarity")
]

for idx, (item_num, label) in enumerate(examples):
    prompt_preds = np.load(WORK_DIR / 'predictions/prompt' / f'item_{item_num:02d}.npy')
    response_preds = np.load(WORK_DIR / 'predictions/response' / f'item_{item_num:02d}.npy')
    
    # Mean activation over vertices
    prompt_mean = np.mean(prompt_preds, axis=1) if prompt_preds.size > 0 else np.array([0])
    response_mean = np.mean(response_preds, axis=1) if response_preds.size > 0 else np.array([0])
    
    time_prompt = np.arange(len(prompt_mean))
    time_response = np.arange(len(response_mean))
    
    # Plot prompt
    axes[idx, 0].plot(time_prompt, prompt_mean, color='steelblue', linewidth=1.5)
    axes[idx, 0].fill_between(time_prompt, prompt_mean, alpha=0.3, color='steelblue')
    axes[idx, 0].set_title(f'Item {item_num}: Prompt ({label})', fontsize=11, fontweight='bold')
    axes[idx, 0].set_ylabel('Mean Brain Activation')
    axes[idx, 0].grid(True, alpha=0.3)
    
    # Plot response
    axes[idx, 1].plot(time_response, response_mean, color='coral', linewidth=1.5)
    axes[idx, 1].fill_between(time_response, response_mean, alpha=0.3, color='coral')
    axes[idx, 1].set_title(f'Item {item_num}: Response ({label})', fontsize=11, fontweight='bold')
    axes[idx, 1].set_ylabel('Mean Brain Activation')
    axes[idx, 1].grid(True, alpha=0.3)

for ax in axes[-1]:
    ax.set_xlabel('Time Step (1 TR = 1s)')

plt.suptitle('Brain Activation Over Time: Prompt (left) vs Response (right)', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(WORK_DIR / 'visualizations/brain_timeseries_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"✅ Saved: brain_timeseries_comparison.png")

In [ ]:
# @title 🧠 Spatial Pattern Comparison
print("=" * 60)
print("SPATIAL PATTERN COMPARISON")
print("=" * 60)

fig, axes = plt.subplots(3, 2, figsize=(14, 10))

examples = [
    (top_item, "High Similarity"),
    (low_item, "Low Similarity"),
    (int(segmented_df.iloc[14]['item_number']), "Medium Similarity")
]

for idx, (item_num, label) in enumerate(examples):
    prompt_preds = np.load(WORK_DIR / 'predictions/prompt' / f'item_{item_num:02d}.npy')
    response_preds = np.load(WORK_DIR / 'predictions/response' / f'item_{item_num:02d}.npy')
    
    # Mean over time
    prompt_spatial = np.mean(prompt_preds, axis=0) if prompt_preds.size > 0 else np.zeros(20484)
    response_spatial = np.mean(response_preds, axis=0) if response_preds.size > 0 else np.zeros(20484)
    
    # Plot prompt (first 5000 vertices)
    axes[idx, 0].plot(prompt_spatial[:5000], color='steelblue', alpha=0.7)
    axes[idx, 0].set_title(f'Item {item_num}: Prompt Spatial ({label})', fontsize=11, fontweight='bold')
    axes[idx, 0].set_xlabel('Vertex Index')
    axes[idx, 0].set_ylabel('Mean Activation')
    axes[idx, 0].axhline(0, color='gray', linestyle='--', alpha=0.3)
    axes[idx, 0].grid(True, alpha=0.3)
    
    # Plot response
    axes[idx, 1].plot(response_spatial[:5000], color='coral', alpha=0.7)
    axes[idx, 1].set_title(f'Item {item_num}: Response Spatial ({label})', fontsize=11, fontweight='bold')
    axes[idx, 1].set_xlabel('Vertex Index')
    axes[idx, 1].set_ylabel('Mean Activation')
    axes[idx, 1].axhline(0, color='gray', linestyle='--', alpha=0.3)
    axes[idx, 1].grid(True, alpha=0.3)

plt.suptitle('Brain Spatial Patterns: Prompt (left) vs Response (right)\n(First 5000 vertices shown)', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(WORK_DIR / 'visualizations/brain_spatial_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"✅ Saved: brain_spatial_comparison.png")

## 11. Summary & Conclusions

Final analysis and interpretation of results.

In [ ]:
# @title 📊 Final Summary
print("=" * 60)
print("EXPERIMENT SUMMARY")
print("=" * 60)

print(f"\n📋 Dataset:")
print(f"   Participant: 038010-2A")
print(f"   Items processed: {len(similarity_df)}")
print(f"   Total segments: {len(similarity_df) * 2}")

print(f"\n🧠 TRIBE v2 Model:")
print(f"   Model: facebook/tribev2")
print(f"   Output: fsaverage5 cortical mesh (~20k vertices)")
print(f"   Resolution: 1 TR = 1 second")

print(f"\n📈 Similarity Results:")
print(f"   Cosine Similarity:")
print(f"      Mean: {similarity_df['cosine_similarity'].mean():.4f} ± {similarity_df['cosine_similarity'].std():.4f}")
print(f"      Range: [{similarity_df['cosine_similarity'].min():.4f}, {similarity_df['cosine_similarity'].max():.4f}]")
print(f"\n   Pearson Correlation:")
print(f"      Mean: {similarity_df['pearson_correlation'].mean():.4f} ± {similarity_df['pearson_correlation'].std():.4f}")
print(f"      Range: [{similarity_df['pearson_correlation'].min():.4f}, {similarity_df['pearson_correlation'].max():.4f}]")

print(f"\n🏆 Top Highest Similarity Items:")
for _, row in top5_high.head(3).iterrows():
    print(f"   Item {row['item_number']}: {row['stimulus'][:30]}... → {row['cosine_similarity']:.4f}")

print(f"\n⚠️ Top Lowest Similarity Items:")
for _, row in top5_low.head(3).iterrows():
    print(f"   Item {row['item_number']}: {row['stimulus'][:30]}... → {row['cosine_similarity']:.4f}")

print(f"\n📁 Output Files:")
print(f"   {WORK_DIR}/audio/prompt/ - {len(list((WORK_DIR / 'audio/prompt').glob('*.wav')))} files")
print(f"   {WORK_DIR}/audio/response/ - {len(list((WORK_DIR / 'audio/response').glob('*.wav')))} files")
print(f"   {WORK_DIR}/predictions/prompt/ - {len(list((WORK_DIR / 'predictions/prompt').glob('*.npy')))} files")
print(f"   {WORK_DIR}/predictions/response/ - {len(list((WORK_DIR / 'predictions/response').glob('*.npy')))} files")
print(f"   {WORK_DIR}/visualizations/ - {len(list((WORK_DIR / 'visualizations').glob('*.png')))} files")

In [ ]:
# @title 🔍 Interpretation & Insights
print("=" * 60)
print("INTERPRETATION & INSIGHTS")
print("=" * 60)

print("""
### Key Findings

1. **Brain-State Similarity Varies**
   - Cosine similarity ranges from {:.2f} to {:.2f}
   - This suggests TRIBE v2 can detect differences in brain activation patterns

2. **High Similarity Items**
   - Items with high similarity may indicate better learner performance
   - Interpretation: The learner's response triggered similar semantic processing

3. **Low Similarity Items**
   - Items with low similarity may indicate:
     - Phonetic errors affecting processing
     - Meaning divergence
     - Different attention/focus

### Limitations

1. **Single Participant** - Results may not generalize
2. **No Ground Truth** - Cannot validate against human scores
3. **Predicted vs Real fMRI** - TRIBE v2 predictions, not actual brain activity
4. **Audio Duration Mismatch** - Prompt/response have different durations

### Next Steps

1. **Multi-Participant Study** - Test on more participants
2. **Human Validation** - Get human raters to score responses
3. **Threshold Calibration** - Map similarity to 0-4 scores
4. **Error Analysis** - Examine specific error patterns
""".format(
similarity_df['cosine_similarity'].min(),
similarity_df['cosine_similarity'].max()
))

In [ ]:
# @title 💾 Download All Results
import shutil
from google.colab import files

print("=" * 60)
print("DOWNLOAD RESULTS")
print("=" * 60)

print("\nTo download results, run:")
print(f"\n1. Download similarity results:")
print(f"   files.download('{WORK_DIR / 'similarity_results.csv'}')")

print(f"\n2. Or download the entire EXP-1 folder:")
print(f"   shutil.make_archive('EXP-1', 'zip', '{WORK_DIR}')")
print(f"   files.download('EXP-1.zip')")

print("\n✅ Notebook complete!")

---

## 🎯 Next Steps

1. **Fill timestamps** - Complete the CSV with your manual annotations
2. **Run notebook** - Upload files and run all cells
3. **Analyze results** - Review the visualizations
4. **Iterate** - Adjust timestamps if needed

---

*Experiment 1 (EXP-1) - AutoEIT + TRIBE v2*